In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '7'

In [2]:
import torch
from torchmetrics import StructuralSimilarityIndexMeasure, MeanSquaredError
from torchmetrics.multimodal.clip_score import CLIPScore
from torchmetrics.image.fid import FrechetInceptionDistance
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset

from PIL import Image
from torch.nn import functional as F
from tqdm import tqdm
import json

In [3]:
img_dir = '/amax/hchuz/Image-to-Graph/dataset/original'
res_dir = '/amax/hchuz/architectural_heritage/results/imgs_eval_200_focus'
names = [name[:-len('.png')] for name in os.listdir(img_dir) if name.endswith('.png')]
result_dict = {name: {'path':os.path.join(img_dir, name+'.png'), 'result_list':[]} for name in names}
for res in os.listdir(res_dir):
    if 'control' in res:
        continue
    for name in result_dict.keys():
        if name in res:
            result_dict[name]['result_list'].append(os.path.join(res_dir, res))
            break
result_dict

{'Snipaste_2023-02-21_12-59-54': {'path': '/amax/hchuz/Image-to-Graph/dataset/original/Snipaste_2023-02-21_12-59-54.png',
  'result_list': ['/amax/hchuz/architectural_heritage/results/imgs_eval_200_focus/Snipaste_2023-02-21_12-59-54_111.png',
   '/amax/hchuz/architectural_heritage/results/imgs_eval_200_focus/Snipaste_2023-02-21_12-59-54_222.png',
   '/amax/hchuz/architectural_heritage/results/imgs_eval_200_focus/Snipaste_2023-02-21_12-59-54_333.png',
   '/amax/hchuz/architectural_heritage/results/imgs_eval_200_focus/Snipaste_2023-02-21_12-59-54_444.png']},
 'Snipaste_2023-02-21_13-01-28': {'path': '/amax/hchuz/Image-to-Graph/dataset/original/Snipaste_2023-02-21_13-01-28.png',
  'result_list': ['/amax/hchuz/architectural_heritage/results/imgs_eval_200_focus/Snipaste_2023-02-21_13-01-28_111.png',
   '/amax/hchuz/architectural_heritage/results/imgs_eval_200_focus/Snipaste_2023-02-21_13-01-28_222.png',
   '/amax/hchuz/architectural_heritage/results/imgs_eval_200_focus/Snipaste_2023-02-21_1

In [4]:
class ImagePairDataset(Dataset):
    def __init__(self, result_dict, normalize):
        self.image_path_pairs = []
        for v in result_dict.values():
            path = v['path']
            for res_path in v['result_list']:
                self.image_path_pairs.append((path, res_path))
        #self.result_dict = result_dict
        self.transform = transforms.ToTensor()
        self.normalize = normalize

    def __len__(self):
        return len(self.image_path_pairs)

    def __getitem__(self, idx):
        ori_img = Image.open(self.image_path_pairs[idx][0]).convert('RGB')
        res_img = Image.open(self.image_path_pairs[idx][1]).convert('RGB')
        res_img = res_img.resize(ori_img.size)
        ori_img = self.transform(ori_img)
        res_img = self.transform(res_img)
        if not self.normalize:
            ori_img = (ori_img * 255).to(torch.uint8)
            res_img = (res_img * 255).to(torch.uint8)
        return ori_img, res_img

In [5]:
def get_dataset_dataloader(batch_size, normalize = True):
    dataset = ImagePairDataset(result_dict, normalize=normalize)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    return dataset, dataloader

In [6]:
batch_size = 4

In [7]:
ssim = StructuralSimilarityIndexMeasure(data_range=1.0)
normalize = True
dataset, dataloader = get_dataset_dataloader(batch_size=batch_size, normalize=True)
sum = 0
count = 0
for batch in tqdm(dataloader):
    sum += ssim(batch[0], batch[1])
    count += 1
ssim_res = sum.item() / count
ssim_res

/amax/hchuz/miniconda3/envs/py310/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:62: FutureWarning: Importing `StructuralSimilarityIndexMeasure` from `torchmetrics` was deprecated and will be removed in 2.0. Import `StructuralSimilarityIndexMeasure` from `torchmetrics.image` instead.
  _future_warning(
100%|██████████| 192/192 [01:30<00:00,  2.11it/s]


0.33338868618011475

In [8]:
mse = MeanSquaredError()
dataset, dataloader = get_dataset_dataloader(batch_size=batch_size, normalize=True)
sum = 0
count = 0
for batch in tqdm(dataloader):
    sum += mse(batch[0], batch[1])
    count += 1
mse_res = sum.item() / count
mse_res = 100*(1-mse_res)
mse_res

100%|██████████| 192/192 [00:40<00:00,  4.70it/s]


93.4592699011167

In [9]:
fid = FrechetInceptionDistance(feature=64, normalize=True)
dataset, dataloader = get_dataset_dataloader(batch_size=batch_size, normalize=True)
for batch in tqdm(dataloader):
    fid.update(batch[0], real=True)
    fid.update(batch[1], real=False)
fid_res = fid.compute().item()
fid_res

100%|██████████| 192/192 [00:47<00:00,  4.03it/s]


1.06917405128479

In [10]:
clip_score = CLIPScore(model_name_or_path="openai/clip-vit-base-patch16").to('cuda')
dataset, dataloader = get_dataset_dataloader(batch_size=4, normalize=False)
sum = 0
count = 0
for batch in tqdm(dataloader):
    score = clip_score(batch[0], batch[1])
    sum += score
    count += 1
clip_res = sum.item() / count
clip_res

100%|██████████| 192/192 [03:33<00:00,  1.11s/it]


82.06274922688802

In [11]:
eval_dict = {'img_dir': img_dir, 'res_dir': res_dir, 'ssim': ssim_res, 'mse': mse_res, 'fid': fid_res, 'clip': clip_res}
with open(os.path.join(res_dir, 'result.json'), 'w', encoding='utf-8') as f:
    json.dump(eval_dict, f, ensure_ascii=False, indent=2)